# 01 — Vérification et création de splits équilibrés

Objectif : corriger le problème où seul `dbpedia_14` semble être utilisé, puis produire des splits **balanced** reproductibles pour tous les datasets.

Ce notebook :
- clone le repo de Georges au début ;
- charge les datasets via `repeng` ;
- sélectionne uniquement des splits équilibrés `True/False` ;
- force la même taille de train pour chaque dataset et la même taille de validation pour chaque dataset ;
- sauvegarde un manifeste JSON réutilisable par le notebook grand modèle ;
- ajoute des assertions qui arrêtent le notebook si un dataset manque.

In [ ]:
# Cellule demandée : clone du repo de Georges + installation minimale
%pip install -q datasets pydantic jsonlines tqdm scikit-learn pandas matplotlib jaxtyping overrides
%pip install -q "mppr @ git+https://github.com/mishajw/mppr.git@main"

from pathlib import Path
import os, sys, subprocess, json, random, hashlib

WORKDIR = Path.cwd()
GEORGE_REPO = WORKDIR / "llm_lie_detection_black_vs_white_box"

if not GEORGE_REPO.exists():
    !git clone https://github.com/GeorgeVasile04/llm_lie_detection_black_vs_white_box.git
else:
    print(f"Repo déjà présent: {GEORGE_REPO}")

sys.path.insert(0, str(GEORGE_REPO.resolve()))
sys.path.insert(0, str((GEORGE_REPO / "White_Box_Lie_Detection").resolve()))

In [ ]:
# Imports + configuration
import numpy as np
import pandas as pd
from collections import Counter, defaultdict
from pathlib import Path
from tqdm.auto import tqdm

try:
    import overrides
    overrides.override = lambda f=None, *args, **kwargs: (f if f is not None else (lambda x: x))
except Exception:
    pass

from White_Box_Lie_Detection.repeng.datasets.elk.utils.fns import get_dataset

DATASETS = [
    "arc_easy",
    "boolq",
    "common_sense_qa",
    "dbpedia_14",
    "imdb",
]

# Demandé: balanced uniquement.
# La taille finale sera automatiquement réduite si un dataset n'a pas assez d'exemples.
REQUESTED_TRAIN_PER_CLASS = 75
REQUESTED_VAL_PER_CLASS = 75
RANDOM_SEED = 42

OUTPUT_DIR = WORKDIR / "outputs" / "balanced_splits"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_PATH = OUTPUT_DIR / "balanced_splits.json"
SUMMARY_PATH = OUTPUT_DIR / "balanced_splits_summary.csv"

In [ ]:
def row_to_record(dataset_id, key, row):
    return {
        "dataset_id": dataset_id,
        "key": str(key),
        "split": str(row.split),
        "text": str(row.text),
        "label": bool(row.label),
        "group_id": None if row.group_id is None else str(row.group_id),
        "answer_type": None if row.answer_type is None else str(row.answer_type),
    }


def load_records(dataset_id):
    raw = get_dataset(dataset_id)
    records = [row_to_record(dataset_id, key, row) for key, row in raw.items()]
    if not records:
        raise ValueError(f"Dataset vide: {dataset_id}")
    labels = Counter(r["label"] for r in records)
    if len(labels) != 2:
        raise ValueError(f"Dataset non binaire ou déséquilibré impossible: {dataset_id} -> {labels}")
    return records


def min_count_by_label(records):
    c = Counter(r["label"] for r in records)
    return min(c.get(False, 0), c.get(True, 0))


def balanced_sample(records, n_per_class, rng):
    by_label = {False: [], True: []}
    for r in records:
        by_label[bool(r["label"])].append(r)
    for label in [False, True]:
        if len(by_label[label]) < n_per_class:
            raise ValueError(f"Pas assez d'exemples label={label}: {len(by_label[label])} < {n_per_class}")
        rng.shuffle(by_label[label])
    selected = by_label[False][:n_per_class] + by_label[True][:n_per_class]
    rng.shuffle(selected)
    return selected


def make_split_for_dataset(dataset_id, records, n_train_per_class, n_val_per_class, seed=42):
    stable_offset = int(hashlib.md5(dataset_id.encode("utf-8")).hexdigest()[:8], 16)
    rng = random.Random(seed + stable_offset)

    train_pool = [r for r in records if r["split"] == "train"]
    train_rows = balanced_sample(train_pool, n_train_per_class, rng)
    used = {r["key"] for r in train_rows}

    # Priorité à validation/train-hparams ; fallback sur le reste du train si nécessaire.
    val_pool_primary = [
        r for r in records
        if r["key"] not in used and r["split"] in {"validation", "train-hparams"}
    ]
    val_pool_fallback = [
        r for r in records
        if r["key"] not in used and r["split"] == "train"
    ]
    val_pool = list(val_pool_primary)
    if min_count_by_label(val_pool) < n_val_per_class:
        val_pool = val_pool + val_pool_fallback

    val_rows = balanced_sample(val_pool, n_val_per_class, rng)

    # Assertions importantes : balanced + pas de fuite train/validation.
    assert len({r["key"] for r in train_rows}.intersection({r["key"] for r in val_rows})) == 0
    assert Counter(r["label"] for r in train_rows)[False] == Counter(r["label"] for r in train_rows)[True]
    assert Counter(r["label"] for r in val_rows)[False] == Counter(r["label"] for r in val_rows)[True]
    return {"train": train_rows, "validation": val_rows}

In [ ]:
# Chargement des datasets et calcul d'une taille commune compatible avec tous les datasets.
all_records = {}
raw_summary = []

for ds in tqdm(DATASETS, desc="Chargement datasets"):
    records = load_records(ds)
    all_records[ds] = records
    by_split = Counter(r["split"] for r in records)
    by_label = Counter(r["label"] for r in records)
    train_cap = min_count_by_label([r for r in records if r["split"] == "train"])
    raw_summary.append({
        "dataset": ds,
        "total_rows": len(records),
        "raw_false": by_label.get(False, 0),
        "raw_true": by_label.get(True, 0),
        "train_split_rows": by_split.get("train", 0),
        "validation_split_rows": by_split.get("validation", 0),
        "train_hparams_rows": by_split.get("train-hparams", 0),
        "train_cap_per_class": train_cap,
    })

raw_df = pd.DataFrame(raw_summary)
display(raw_df)

n_train_per_class = min(REQUESTED_TRAIN_PER_CLASS, int(raw_df["train_cap_per_class"].min()))
if n_train_per_class < 1:
    raise ValueError("Impossible de créer un train équilibré commun: aucun exemple train suffisant.")

# On cherche la plus grande validation équilibrée commune après avoir réservé le train.
n_val_per_class = min(REQUESTED_VAL_PER_CLASS, n_train_per_class)
while n_val_per_class > 0:
    ok = True
    for ds, records in all_records.items():
        try:
            _ = make_split_for_dataset(ds, records, n_train_per_class, n_val_per_class, RANDOM_SEED)
        except Exception as e:
            ok = False
            break
    if ok:
        break
    n_val_per_class -= 1

if n_val_per_class < 1:
    raise ValueError("Impossible de créer une validation équilibrée commune.")

print(f"Taille finale commune: train={2*n_train_per_class} lignes/dataset "
      f"({n_train_per_class}/classe), validation={2*n_val_per_class} lignes/dataset "
      f"({n_val_per_class}/classe)")

In [ ]:
# Création effective des splits + sauvegarde du manifeste.
splits = {}
summary = []

for ds, records in all_records.items():
    split = make_split_for_dataset(ds, records, n_train_per_class, n_val_per_class, RANDOM_SEED)
    splits[ds] = split
    for split_name in ["train", "validation"]:
        rows = split[split_name]
        counts = Counter(r["label"] for r in rows)
        source_splits = Counter(r["split"] for r in rows)
        summary.append({
            "dataset": ds,
            "split": split_name,
            "n_rows": len(rows),
            "n_false": counts.get(False, 0),
            "n_true": counts.get(True, 0),
            "source_splits": dict(source_splits),
        })

summary_df = pd.DataFrame(summary)
display(summary_df)

# Assertions anti-bug: si seul dbpedia est pris, on échoue ici.
assert set(splits.keys()) == set(DATASETS), f"Datasets manquants: {set(DATASETS) - set(splits.keys())}"
assert len(splits) == len(DATASETS), "Nombre de datasets incorrect"
assert summary_df.groupby("split")["n_rows"].nunique().max() == 1, "Taille non constante entre datasets"
assert (summary_df["n_false"] == summary_df["n_true"]).all(), "Un split n'est pas balanced"

manifest = {
    "config": {
        "datasets": DATASETS,
        "random_seed": RANDOM_SEED,
        "train_per_class": n_train_per_class,
        "validation_per_class": n_val_per_class,
        "train_rows_per_dataset": 2 * n_train_per_class,
        "validation_rows_per_dataset": 2 * n_val_per_class,
        "balanced_only": True,
    },
    "datasets": splits,
}

MANIFEST_PATH.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
summary_df.to_csv(SUMMARY_PATH, index=False)

print(f"Manifeste sauvegardé: {MANIFEST_PATH}")
print(f"Résumé sauvegardé: {SUMMARY_PATH}")

In [ ]:
# Aperçu rapide: vérifier que les textes viennent bien de plusieurs datasets.
for ds in DATASETS:
    print("\n" + "="*80)
    print(ds)
    print("Train:", len(splits[ds]["train"]), "Validation:", len(splits[ds]["validation"]))
    for r in splits[ds]["train"][:2]:
        label = "TRUE" if r["label"] else "FALSE"
        print(f"[{label}] {r['text'][:220].replace(chr(10), ' ')}...")